# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore, process, and analyze the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library. The data covers protein abundance and modifications in extracellular vesicles derived from stimulated human mast cells.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) served at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and records using the `mlcroissant` library. The dataset is defined by a Croissant JSON-LD file at a public URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Let's examine the available record sets, their `@id`s, and their fields. 
All record set, field, and column references are made using their Croissant `@id` for consistency and interoperability.

We'll print a summary of all record sets and their fields.

In [ ]:
# Find and list all record sets and their fields by @id
print("All available record sets and their fields (by @id):\n")
record_sets = [] # list of @id strings
for rs in metadata.record_sets:
    print(f"Record set: @id = {rs.id}, name = {rs.name}")
    record_sets.append(rs.id)
    for field in rs.fields:
        print(f"  Field: @id = {field.id}, name = {field.name}")
    print("")
if not record_sets:
    print("No record sets found in metadata.")

## 3. Data Extraction

Now, let's load data from one or more record sets. The `records()` function allows us to iterate over records in a specific record set, referenced by its Croissant `@id`. We'll load each into a DataFrame for exploratory analysis.

In [ ]:
# Determine available record set IDs
if not record_sets:
    raise RuntimeError("No record sets available to extract.")
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading data from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print("  No records found.")

# For demonstration, set the first record set as the primary one for later EDA
primary_record_set_id = record_sets[0]
primary_df = dataframes[primary_record_set_id]


## 4. Exploratory Data Analysis (EDA)

Let's pick a numeric field from the primary record set to analyze. 
We'll filter records where this field exceeds a threshold, normalize the values, and group by a categorical field if present. All field/column `@id`s are referenced in code comments for clarity.


In [ ]:
# List available columns
print("Available columns for EDA:", primary_df.columns.tolist())

# Manually select a numeric field by Croissant @id (update as appropriate)
# Example: Suppose '@id' of the field is 'coverage_percent'
numeric_field = None
group_field = None
for col in primary_df.columns:
    if 'coverage' in col.lower() or 'percent' in col.lower():
        numeric_field = col
    if 'sample' in col.lower() or 'label' in col.lower():
        group_field = col
if numeric_field is None:
    # Fallback: select the first numeric-like column
    for col in primary_df.columns:
        if pd.api.types.is_numeric_dtype(primary_df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise RuntimeError("Could not auto-detect a numeric field for EDA.")

print(f"Selected numeric field for EDA is: {numeric_field}")

# Filtering
try:
    threshold = float(primary_df[numeric_field].quantile(0.75)) # Example: use Q3 as threshold
except Exception as e:
    threshold = 10
filtered_df = primary_df[pd.to_numeric(primary_df[numeric_field], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping (if group field exists)
if group_field and group_field in filtered_df.columns:
    grouped_df = (
        filtered_df
        .groupby(group_field)
        [[numeric_field, f"{numeric_field}_normalized"]]
        .mean()
    )
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and the result of the normalization process.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(pd.to_numeric(primary_df[numeric_field], errors='coerce').dropna(), bins=20, ax=ax[0], color='skyblue')
ax[0].set_title(f"Distribution of {numeric_field}")

if f"{numeric_field}_normalized" in filtered_df.columns:
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=20, ax=ax[1], color='orange')
    ax[1].set_title(f"Normalized {numeric_field} in filtered data")
else:
    ax[1].text(0.5, 0.5, "No normalized data", ha='center', va='center')
plt.tight_layout()
plt.show()

## 6. Conclusion

- We successfully loaded the FAIR^2 dataset with `mlcroissant`.
- We inspected the dataset's structure (record sets and fields using their Croissant `@id`).
- We performed simple exploratory data analysis, including filtering, normalization, and grouping, using only the standardized Croissant entity references.
- We visualized the distribution of a key numeric field, aiding further data science and bioinformatics workflows.

**Next steps:** You can continue by engineering features, training models, or integrating with laboratory/clinical frameworks leveraging machine-readable metadata provided by the Croissant format.
